# Global registration with RANSAC
We are going to use [open3d](http://www.open3d.org/) to handle point clouds and generation of point clouds
We are importing the packages and defining a function which helps us drawing the point clouds.

In [1]:
import open3d as o3d
import numpy as np
import copy

# helper function for drawing
# If you want it to be more clear set recolor=True
def draw_registrations(source, target, transformation = None, recolor = False):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    if(recolor):
        source_temp.paint_uniform_color([1, 0.706, 0])
        target_temp.paint_uniform_color([0, 0.651, 0.929])
    if(transformation is not None):
        source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp])

We need to read in our pointclouds. For that we use the `io` module of the
open3d package (`o3d`). The following cell will open a window with a
visualization of both point clouds `source` and `target`.

Also, this page [Visualization - Open3D](http://open3d.org/html/tutorial/Basic/visualization.html)
contains some useful examples and instructions on how to use the viewer.

In [2]:
source = o3d.io.read_point_cloud("ICP/r1.pcd")
target = o3d.io.read_point_cloud("ICP/r2.pcd")

# Used for downsampling.
voxel_size = 0.05

# Show models side by side
draw_registrations(source, target)

### Finding features in pointclouds
When working on point clouds it can be beneficial to work on a downsampled version of the point cloud,
as it decreases the need of computation.

You can use [`pointcloud.voxel_down_sample()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample) where `pointcloud` is the name of your point cloud object. In our case, that would be `source` and `target`.

We also need to estimate the normals of the point cloud points using [`pointcloud.estimate_normals()`](http://www.open3d.org/docs/latest/python_api/open3d.geometry.PointCloud.html#open3d.geometry.PointCloud.voxel_down_sample)

**Task:** Find FPFH features or correspondances of the downsampled point clouds.
[`o3d.pipelines.registration.compute_fpfh_feature()`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.compute_fpfh_feature.html)


In [3]:
# Downsample
source_sample = source.voxel_down_sample(voxel_size)
target_sample = target.voxel_down_sample(voxel_size)

# Estimate normals
source_sample.estimate_normals()
target_sample.estimate_normals()

# Compute FPFH features
source_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    source_sample,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
)
target_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
    target_sample,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
)


### RANSAC 
We will now attempt to use RANSAC to do a global registration of the two point clouds.

By using the function [`o3d.pipelines.registration.registration_ransac_based_on_feature_matching`](http://www.open3d.org/docs/latest/python_api/open3d.pipelines.registration.registration_ransac_based_on_feature_matching.html) from open3d, do the following:


Try to find the transformation from `r1.pcd` (`source`) to `r2.pcd` (`target`).
Attempt with point-to-point and point-to-plane
```Python
point_to_point =  o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane =  o3d.pipelines.registration.TransformationEstimationPointToPlane()
```

When using RANSAC, focus on the arguments below. The rest are optional parameters.
```Python
ransac_result = o3d.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    distance_threshold,
    point_to_point)
```

In [4]:
# Define transformation estimation methods
point_to_point = o3d.pipelines.registration.TransformationEstimationPointToPoint(False)
point_to_plane = o3d.pipelines.registration.TransformationEstimationPointToPlane()

# Set distance threshold
distance_threshold = voxel_size * 1.5

# Call RANSAC with point-to-point estimation
ransac_result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample,
    source_fpfh, target_fpfh,
    True,
    distance_threshold,
    point_to_point)

# Visualize the result
draw_registrations(source, target, ransac_result.transformation, True)

## Exercises
### A)
Can you get a decent transformation from r1 to r3? (check the ICP folder)
### B)
With the following checkers, can you get better results from RANSAC? Try tweaking the parameters of them. Can you make point-to-plane work? Do not spend too much time on this, if you can't manage, skip it. (I was not able to get a good fit.)

You can also try tweaking the `voxel_size`

```Python
corr_length = 0.9
distance_threshold = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0,c1,c2]

ransac_result = o3d.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample, 
    source_fpfh, target_fpfh, 
    True,
    distance_threshold,
    point_to_point,
    checkers = checker_list)
```


## Solution to Exercise A: Transform r1 to r3

In [5]:
# Load r1 and r3
source_r3 = o3d.io.read_point_cloud("ICP/r1.pcd")
target_r3 = o3d.io.read_point_cloud("ICP/r3.pcd")

# Downsample
source_sample_r3 = source_r3.voxel_down_sample(voxel_size)
target_sample_r3 = target_r3.voxel_down_sample(voxel_size)

# Estimate normals
source_sample_r3.estimate_normals()
target_sample_r3.estimate_normals()

# Compute FPFH features
source_fpfh_r3 = o3d.pipelines.registration.compute_fpfh_feature(
    source_sample_r3,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
)
target_fpfh_r3 = o3d.pipelines.registration.compute_fpfh_feature(
    target_sample_r3,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size * 5, max_nn=100)
)

# Run RANSAC for r1 to r3
ransac_result_r3 = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample_r3, target_sample_r3,
    source_fpfh_r3, target_fpfh_r3,
    True,
    distance_threshold,
    point_to_point)

print(f"Fitness: {ransac_result_r3.fitness:.4f}")
print(f"RMSE: {ransac_result_r3.inlier_rmse:.4f}")

# Visualize the result
draw_registrations(source_r3, target_r3, ransac_result_r3.transformation, True)

Fitness: 0.7750
RMSE: 0.0331


## Solution to Exercise B: Improve RANSAC with correspondence checkers

We'll try multiple parameter combinations to see if we can improve the results.

In [6]:
# Experiment 1: Basic checkers with point-to-point
print("Experiment 1: Point-to-Point with correspondence checkers")
corr_length = 0.9
distance_threshold_b = voxel_size * 1.5

c0 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(corr_length)
c1 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold_b)
c2 = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.095)

checker_list = [c0, c1, c2]

ransac_b1 = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample,
    source_fpfh, target_fpfh,
    True,
    distance_threshold_b,
    point_to_point,
    checkers=checker_list)

print(f"Fitness: {ransac_b1.fitness:.4f}, RMSE: {ransac_b1.inlier_rmse:.4f}")
draw_registrations(source, target, ransac_b1.transformation, True)

Experiment 1: Point-to-Point with correspondence checkers
Fitness: 0.6758, RMSE: 0.0276


In [7]:
# Experiment 2: Point-to-Plane with correspondence checkers
print("Experiment 2: Point-to-Plane with correspondence checkers")

ransac_b2 = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample,
    source_fpfh, target_fpfh,
    True,
    distance_threshold_b,
    point_to_plane,
    checkers=checker_list)

print(f"Fitness: {ransac_b2.fitness:.4f}, RMSE: {ransac_b2.inlier_rmse:.4f}")
draw_registrations(source, target, ransac_b2.transformation, True)

Experiment 2: Point-to-Plane with correspondence checkers
Fitness: 0.0000, RMSE: 0.0000


In [8]:
# Experiment 3: Tweak voxel_size and parameters
print("Experiment 3: Smaller voxel size (0.03) with adjusted parameters")
voxel_size_exp = 0.03

# Re-downsample with smaller voxel size
source_exp = source.voxel_down_sample(voxel_size_exp)
target_exp = target.voxel_down_sample(voxel_size_exp)

source_exp.estimate_normals()
target_exp.estimate_normals()

source_fpfh_exp = o3d.pipelines.registration.compute_fpfh_feature(
    source_exp,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_exp * 5, max_nn=100)
)
target_fpfh_exp = o3d.pipelines.registration.compute_fpfh_feature(
    target_exp,
    o3d.geometry.KDTreeSearchParamHybrid(radius=voxel_size_exp * 5, max_nn=100)
)

distance_threshold_exp = voxel_size_exp * 2.0
c0_exp = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.85)
c1_exp = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(distance_threshold_exp)
c2_exp = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.1)

checker_list_exp = [c0_exp, c1_exp, c2_exp]

ransac_b3 = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_exp, target_exp,
    source_fpfh_exp, target_fpfh_exp,
    True,
    distance_threshold_exp,
    point_to_point,
    checkers=checker_list_exp)

print(f"Fitness: {ransac_b3.fitness:.4f}, RMSE: {ransac_b3.inlier_rmse:.4f}")
draw_registrations(source, target, ransac_b3.transformation, True)

Experiment 3: Smaller voxel size (0.03) with adjusted parameters
Fitness: 0.6570, RMSE: 0.0176


In [ ]:
# Experiment 4: Relaxed checkers for point-to-plane
print("Experiment 4: Point-to-Plane with relaxed checker parameters")

# Try more relaxed parameters
c0_relaxed = o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(0.95)
c1_relaxed = o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(voxel_size * 2.5)
c2_relaxed = o3d.pipelines.registration.CorrespondenceCheckerBasedOnNormal(0.15)

checker_list_relaxed = [c0_relaxed, c1_relaxed, c2_relaxed]

ransac_b4 = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
    source_sample, target_sample,
    source_fpfh, target_fpfh,
    True,
    voxel_size * 2.0,
    point_to_plane,
    checkers=checker_list_relaxed)

print(f"Fitness: {ransac_b4.fitness:.4f}, RMSE: {ransac_b4.inlier_rmse:.4f}")
draw_registrations(source, target, ransac_b4.transformation, True)

Experiment 4: Point-to-Plane with relaxed checker parameters
Fitness: 0.0000, RMSE: 0.0000


: 